In [1]:
import pandas as pd
import numpy as np

# ── Paramètres globaux ──────────────────────────────────────────────────────
np.random.seed(42)
n_clients   = 300
start_date  = pd.to_datetime("2020-01-01")
date_extraction = pd.Timestamp("2025-01-01")


# ── Amélioration 1 : Facture mensuelle par distribution continue ─────────────
#chaque client reçoit une valeur tirée dans l'intervalle de son segment
def generer_facture():
    segment = np.random.choice([0, 1, 2], p=[0.70, 0.20, 0.10])
    bornes   = [(5, 25), (30, 60), (70, 160)]
    return np.round(np.random.uniform(*bornes[segment]), 2)


# ── Amélioration 2 : Cohérence âge / date d'abonnement ─────────────────────
#on rejette et on retire une date de naissance si le client avait < 18 ans à la date de début d'abonnement
start_birth = pd.to_datetime("1945-01-01")
end_birth   = pd.to_datetime("2007-01-01")
diff_days   = (end_birth - start_birth).days

dates_abo_fixes = [
    start_date + pd.Timedelta(days=np.random.randint(0, 1461))
    for _ in range(n_clients)
]

dates_naissance = []
for date_abo in dates_abo_fixes:
    while True:
        dn = start_birth + pd.Timedelta(days=np.random.randint(0, diff_days))
        age_a_souscription = (date_abo - dn).days / 365.25
        if age_a_souscription >= 18:
            dates_naissance.append(dn)
            break


# ── Variables pré-calculées ────────────────────
satisfaction_initiale = np.random.randint(1, 6, n_clients)
statut_actif  = np.random.choice([True, False], n_clients, p=[0.7, 0.3])
consentement  = np.random.choice([True, False], n_clients, p=[0.6, 0.4])
optout        = [
    np.random.choice([True, False], p=[0.2, 0.8]) if c else False
    for c in consentement
]


# ── Construction du DataFrame clients ───────────────────────────────────────
clients = pd.DataFrame({
    "client_id"       : range(1, n_clients + 1),
    "nom_client"      : [f"Client_{i}" for i in range(1, n_clients + 1)],
    "genre_client"    : np.random.choice(["Homme", "Femme"], n_clients),
    "date_naissance"  : dates_naissance,

    "adresse_email"   : [
        f"client_{i}@{np.random.choice(['gmail.com','outlook.fr','yahoo.com','tunisietelecom.tn'])}"
        for i in range(1, n_clients + 1)
    ],

    "num_tel_mobile"  : [
        f"+216{np.random.choice([2,4,5,9])}{np.random.randint(1000000, 9999999)}"
        for _ in range(n_clients)
    ],

    "adresse_physique": [f"Adresse_{i}" for i in range(1, n_clients + 1)],

    # ── Amélioration 3 : format CIN corrigé ─────────────────────────────────
    # Avant : randint(10000000, 99999999) produit 8 chiffres → :07d inutile → 9 chars
    # Après : randint(1000000, 9999999) produit 7 chiffres → CIN = 1 + 7 = 8 chars
    "identifiant_national": [
        f"{np.random.randint(0, 2)}{np.random.randint(1000000, 9999999):07d}"
        for _ in range(n_clients)
    ],

    "type_abonnement" : np.random.choice(
        ["Offre Prépayée", "Offre à Facture"], n_clients, p=[0.85, 0.15]
    ),
    "plan_tarifaire"  : np.random.choice(
        ["Offre Classique", "Forfait Mobile (Mixte)", "Forfait Illimité"],
        n_clients, p=[0.70, 0.20, 0.10]
    ),
    "date_debut_abonnement": dates_abo_fixes,
    "statut_actif"    : statut_actif,
    "moyen_paiement"  : np.random.choice(
        ["ticket_recharge", "especes", "prelevement_bancaire"],
        n_clients, p=[0.75, 0.15, 0.10]
    ),

    # Amélioration 1 appliquée ici
    "facture_moyenne_mensuelle": [generer_facture() for _ in range(n_clients)],

    # Satisfaction basse pour les clients inactifs (churned)
    "satisfaction_client": [
        s if actif else np.random.randint(1, 3)
        for s, actif in zip(satisfaction_initiale, statut_actif)
    ],

    # ── Amélioration 4 : date_consentement = NaT si pas de consentement ─────
    #NaT pour les clients avec consentement_marketing = False
    "date_consentement": [
        d + pd.Timedelta(days=np.random.randint(0, 365)) if c else pd.NaT
        for d, c in zip(dates_abo_fixes, consentement)
    ],

    "consentement_marketing": consentement,
    "optout_marketing"      : optout,
})


# ── Amélioration 5 : tenure avec diviseur réel (30.44 j/mois) ───────────────
#365.25 / 12 = 30.44 jours par mois en moyenne
clients["tenure_jours"] = (date_extraction - clients["date_debut_abonnement"]).dt.days
clients["tenure_mois"]  = (clients["tenure_jours"] / 30.44).astype(int)


# ── Amélioration 6 : variable cible churn encodée en 0 / 1 ─────────────────
# Avant : statut_actif (booléen) jamais converti → dataset sans variable cible
# Après : churn = 1 si le client a quitté l'opérateur, 0 s'il est encore actif

print(f"Dataset clients : {clients.shape}")
print(clients[["client_id","facture_moyenne_mensuelle","tenure_mois"]].head())

Dataset clients : (300, 20)
   client_id  facture_moyenne_mensuelle  tenure_mois
0          1                       5.17           23
1          2                       9.15           12
2          3                       9.39           31
3          4                      21.53           17
4          5                      12.07           22


Ce bloc construit la table centrale du dataset avec 300 clients simulés. Plusieurs choix méthodologiques ont été appliqués pour garantir le réalisme et la cohérence des données :

Reproductibilité : np.random.seed(42) assure que chaque exécution produit exactement les mêmes données, condition indispensable en contexte de recherche et de ML.

Facture mensuelle : générée par une fonction qui tire une valeur continue dans l'intervalle de chaque segment tarifaire (bas / moyen / premium), évitant la réduction à 3 valeurs distinctes.

Cohérence âge / contrat : une boucle de validation garantit que chaque client avait au moins 18 ans à la date de souscription.

Identifiant national : corrigé pour respecter le format CIN tunisien à 8 chiffres.

Date de consentement : assignée à NaT pour les clients n'ayant pas donné leur accord marketing — respecte la règle métier RGPD-like.

Tenure : calculée avec un diviseur de 30.44 jours/mois (moyenne réelle) plutôt que 30.

In [27]:
# ── Génération des événements CDR (appels, SMS, données mobiles) ─────────────

events = []

# ── Amélioration 5 : multiplicateurs par plan tarifaire ─────────────────────
# Définit le comportement moyen selon l'offre souscrite
# λ_base  = volume d'événements moyen
# data_mu = paramètre mean de la lognormale pour la consommation data
plan_config = {
    "Offre Classique"        : {"lambda_base": 20, "data_mu": 1.5},
    "Forfait Mobile (Mixte)" : {"lambda_base": 30, "data_mu": 2.0},
    "Forfait Illimité"       : {"lambda_base": 45, "data_mu": 2.8},
}

for _, c in clients.iterrows():

    config = plan_config[c.plan_tarifaire]

    # ── Amélioration 1 : volume d'événements conditionné au statut ───────────
    #λ réduit pour les clients churné (activité naturellement plus faible)
    if c.statut_actif:
        n_ev = np.random.poisson(config["lambda_base"])
    else:
        n_ev = np.random.poisson(max(1, config["lambda_base"] // 5))

    for _ in range(n_ev):

        # ── Amélioration 2 : date_extraction au lieu du timestamp hardcodé ───
        #utilisation de la variable globale date_extraction
        delta_max = (date_extraction - c.date_debut_abonnement).days
        if delta_max <= 0:
            delta_max = 1

        timestamp  = c.date_debut_abonnement + pd.to_timedelta(
            np.random.randint(0, delta_max), unit="d"
        )
        timestamp += pd.to_timedelta(np.random.randint(0, 86400), unit="s")

        type_ev = np.random.choice(
            ["appel", "sms", "donnee_mobile"], p=[0.4, 0.3, 0.3]
        )

        # ── Amélioration 3 : numéro destination généré uniquement si pertinent
        # Avant : num_dest calculé pour tous les types, même donnee_mobile
        # Après : calcul conditionnel → 0 opérations inutiles sur les sessions data
        if type_ev != "donnee_mobile":
            dest_prefix = np.random.choice([2, 4, 5, 9])
            num_dest    = f"+216{dest_prefix}{np.random.randint(1000000, 9999999)}"
        else:
            num_dest = "INTERNET"

        ev = {
            "client_id"          : c.client_id,
            "date_heure"         : timestamp,
            "type_evenement"     : type_ev,
            "numero_source"      : c.num_tel_mobile,
            "numero_destination" : num_dest,
            "duree_appel_sec"    : 0,
            "sms_compte"         : 0,
            "data_mb"            : 0,
        }

        if type_ev == "appel":
            # Loi exponentielle : beaucoup d'appels courts, quelques très longs
            # scale=180 → durée moyenne de 3 minutes, réaliste pour un mobile
            ev["duree_appel_sec"] = int(np.random.exponential(scale=180))

        elif type_ev == "sms":
            ev["sms_compte"] = 1

        else:
            # ── Amélioration 4 : paramètres lognormale corrigés ─────────────
            #mean conditionné au plan, sigma=1 → médiane ~7–16 MB/session
            ev["data_mb"] = np.round(
                np.random.lognormal(mean=config["data_mu"], sigma=1.0), 2
            )

        events.append(ev)

usage = pd.DataFrame(events)

# ── Amélioration 6 : tri chronologique ──────────────────────────────────────
# Nécessaire pour toute analyse temporelle : récence, tendance, déclin avant churn
usage = usage.sort_values("date_heure").reset_index(drop=True)

# ── Vérification rapide ──────────────────────────────────────────────────────
print(f"Événements CDR générés : {len(usage)}")
print(f"Répartition des types  :\n{usage['type_evenement'].value_counts()}")
print(usage.head())

Événements CDR générés : 5388
Répartition des types  :
type_evenement
appel            2149
sms              1647
donnee_mobile    1592
Name: count, dtype: int64
   client_id          date_heure type_evenement numero_source  \
0        203 2020-01-05 08:29:01  donnee_mobile  +21694471750   
1        259 2020-01-26 23:45:34            sms  +21658601723   
2         54 2020-01-31 23:05:08          appel  +21659181883   
3         29 2020-02-04 05:09:55          appel  +21657879771   
4         67 2020-02-05 17:27:00          appel  +21644895958   

  numero_destination  duree_appel_sec  sms_compte  data_mb  
0           INTERNET                0           0      4.8  
1       +21691110629                0           1      0.0  
2       +21627473049               53           0      0.0  
3       +21651558745               60           0      0.0  
4       +21641642006              598           0      0.0  


Ce bloc simule les événements d'usage télécom de chaque client : appels voix, SMS et sessions data mobile. Plusieurs ajustements ont été apportés pour renforcer le réalisme comportemental et la cohérence du pipeline :

Volume d'événements conditionné au statut : un client churné génère environ 5× moins d'événements qu'un client actif, reflétant le déclin d'activité qui précède et accompagne le départ.

Comportement lié au plan tarifaire : le volume moyen d'événements et la consommation data sont modulés selon l'offre souscrite (Classique / Mixte / Illimité), introduisant une hétérogénéité réaliste entre segments de clients.

Paramètres statistiques corrigés : la durée des appels suit une loi exponentielle (scale=180 s, soit ~3 min en moyenne) et la consommation data une loi log-normale dont les paramètres ont été ramenés à des valeurs réalistes (~7–16 MB médian par session selon le plan, contre ~168 MB dans la version initiale).

Numéro destination : généré uniquement pour les appels et SMS, jamais pour les sessions data où la destination est systématiquement "INTERNET".

Référence temporelle centralisée : utilisation de la variable globale date_extraction plutôt qu'une date hardcodée dans la boucle, garantissant la cohérence si le paramètre global est modifié.

Tri chronologique : le DataFrame final est trié par date_heure, condition nécessaire pour les analyses de récence et de tendance d'usage utilisées comme features de churn.

Distribution statistique choisie : loi de Poisson pour le nombre d'événements (phénomène de comptage discret), loi exponentielle pour les durées d'appel (mémoire sans passé), loi log-normale pour la consommation data (asymétrie positive caractéristique des distributions de consommation).

In [28]:
# ── Génération des empreintes numériques (sessions internet) ─────────────────

dispositifs = [
    ("Android",      "Android 13"),
    ("Android",      "Android 12"),
    ("iOS",          "iOS 17"),
    ("iOS",          "iOS 16"),
    ("FeaturePhone", "Proprietary"),
]

# ── Amélioration 3 : probabilités des dispositifs corrigées ─────────────────
#version récente > version ancienne, FeaturePhone augmenté (contexte TT)
p_dispositifs = [0.42, 0.28, 0.08, 0.05, 0.17]


# ── Amélioration 4 : navigateur conditionné au type d'appareil ──────────────
#règles métier respectées — Safari = iOS only, Samsung Browser = Android only
def get_navigateur(type_app):
    if type_app == "iOS":
        return np.random.choice(["Safari", "Chrome"], p=[0.70, 0.30])
    elif type_app == "FeaturePhone":
        return "Navigateur intégré"
    else:  # Android
        return np.random.choice(["Chrome", "Samsung Browser"], p=[0.75, 0.25])


# ── Amélioration 5 : version MyTT conditionnée au type d'appareil ───────────
#FeaturePhone → "Non applicable", smartphones → version pondérée
def get_version_mytt(type_app):
    if type_app == "FeaturePhone":
        return "Non applicable"
    return np.random.choice(["3.1.0", "3.0.5", "2.9.0"], p=[0.65, 0.25, 0.10])


# ── Construction ligne par ligne pour gérer les dépendances entre colonnes ───
sessions_rows = []

for _, c in clients.iterrows():

    # ── Amélioration 1 : nombre de sessions variable lié au statut ───────────
    #Poisson(8) si actif, Poisson(2) si churné — reflète le désengagement
    lam_sessions = 8 if c.statut_actif else 2
    n_sess = max(1, np.random.poisson(lam_sessions))

    # Récupération du consentement marketing du client (pour amélioration 8)
    consent = clients.loc[clients.client_id == c.client_id, "consentement_marketing"].values[0]

    for _ in range(n_sess):

        # Tirage du dispositif
        idx_disp   = np.random.choice(len(dispositifs), p=p_dispositifs)
        type_app, sys_exp = dispositifs[idx_disp]

        # ── Amélioration 2 : timestamp de session ────────────────────────────
        #date aléatoire entre date_debut_abonnement et date_extraction
        delta = (date_extraction - c.date_debut_abonnement).days
        if delta <= 0:
            delta = 1
        ts_session = c.date_debut_abonnement + pd.Timedelta(
            days=int(np.random.randint(0, delta))
        )

        # ── Amélioration 6 : durée session log-normale ───────────────────────
        #log-normale → médiane ≈ 3 min, queue longue réaliste
        duree = int(np.clip(np.random.lognormal(mean=5.2, sigma=1.1), 30, 7200))

        # ── Amélioration 7 : domaines sans répétition, nombre variable ───────
        #entre 1 et 5 domaines uniques par session
        catalogue = [
            "google.com", "facebook.com", "tunisietelecom.tn",
            "tayara.tn", "youtube.com", "instagram.com", "tiktok.com"
        ]
        n_dom   = np.random.randint(1, min(6, len(catalogue) + 1))
        domaines = ";".join(np.random.choice(catalogue, n_dom, replace=False))

        # ── Amélioration 8 : cookies liés au consentement marketing ──────────
        #probabilité plus faible si le client a refusé le consentement
        p_cookie = 0.80 if consent else 0.35
        cookie   = np.random.choice([True, False], p=[p_cookie, 1 - p_cookie])

        sessions_rows.append({
            "client_id"            : c.client_id,
            "date_heure_session"   : ts_session,          # Amélioration 2
            "adresse_IP"           : f"197.{np.random.randint(0,5)}.{np.random.randint(0,255)}.{np.random.randint(1,255)}",
            "type_appareil"        : type_app,
            "systeme_exploitation" : sys_exp,
            "version_app_MyTT"     : get_version_mytt(type_app),   # Amélioration 5
            "duree_session_sec"    : duree,                         # Amélioration 6
            "historique_domaines"  : domaines,                      # Amélioration 7
            "navigateur"           : get_navigateur(type_app),      # Amélioration 4
            "cookies_acceptes"     : cookie,                        # Amélioration 8
        })

sessions = pd.DataFrame(sessions_rows)

# Tri chronologique (cohérence avec le bloc CDR)
sessions = sessions.sort_values("date_heure_session").reset_index(drop=True)

# ── Vérification rapide ──────────────────────────────────────────────────────
print(f"Sessions générées        : {len(sessions)}")
print(f"Sessions par client (moy): {len(sessions)/n_clients:.1f}")
print(f"\nRépartition navigateurs  :\n{sessions['navigateur'].value_counts()}")
print(f"\nRépartition appareils    :\n{sessions['type_appareil'].value_counts()}")

Sessions générées        : 1837
Sessions par client (moy): 6.1

Répartition navigateurs  :
navigateur
Chrome                1023
Samsung Browser        332
Navigateur intégré     309
Safari                 173
Name: count, dtype: int64

Répartition appareils    :
type_appareil
Android         1295
FeaturePhone     309
iOS              233
Name: count, dtype: int64


Ce bloc simule le comportement digital des clients à travers leurs sessions sur l'application MyTT et le navigateur mobile. La logique de génération a été substantiellement enrichie pour assurer la cohérence interne du dataset :

Nombre de sessions variable : modélisé par une loi de Poisson conditionnée au statut du client (λ=8 si actif, λ=2 si churné), reflétant le désengagement progressif des clients sur le point de partir.

Timestamp de session : chaque session est horodatée entre la date de début d'abonnement et la date d'extraction, rendant possible le calcul de features de récence (ex : jours depuis la dernière connexion).

Cohérence appareil / navigateur / app : Safari est exclusivement attribué aux appareils iOS, Samsung Browser aux Android, et les FeaturePhone reçoivent un navigateur intégré générique et ne sont pas associés à l'application MyTT.

Distribution des durées de session : loi log-normale (médiane ≈ 3 min) remplaçant la distribution uniforme initiale, capturant l'asymétrie réelle des comportements de navigation (majorité de sessions courtes, quelques sessions longues).

Historique de navigation : entre 1 et 5 domaines uniques par session tirés sans remise, éliminant les répétitions artificielles du code initial.

Cookies liés au consentement : la probabilité d'acceptation des cookies est modulée par le profil de consentement marketing du client (80% si consentant, 35% sinon), introduisant une cohérence comportementale entre les tables.

Note méthodologique : la construction ligne par ligne (au lieu d'un DataFrame vectorisé) est ici nécessaire car plusieurs colonnes dépendent les unes des autres (navigateur dépend du type d'appareil, cookies dépendent du consentement client). Cette dépendance inter-colonnes ne peut pas être résolue par une génération vectorisée simple.

In [29]:
# ── Génération des localisations ─────────────────────────────────────────────

# ── Amélioration 3 : zones géographiques avec bornes GPS réalistes ───────────
# Chaque zone correspond à une région géographique tunisienne cohérente
#coordonnées tirées dans les bornes de la zone assignée au client
zones_config = {
    "URBAIN"    : {
        "lat": (35.8, 37.2),   # Grand Tunis, Sfax, Sousse, Bizerte
        "lon": (9.8,  10.8),
        "p_signal": [0.60, 0.35, 0.05],   # Excellent / Bon / Faible
    },
    "SUBURBAIN" : {
        "lat": (34.5, 35.8),   # Périphéries, villes moyennes
        "lon": (8.5,  10.5),
        "p_signal": [0.30, 0.50, 0.20],
    },
    "RURAL"     : {
        "lat": (30.0, 34.5),   # Sud, zones intérieures
        "lon": (7.5,   9.5),
        "p_signal": [0.10, 0.40, 0.50],
    },
}

# ── Amélioration 1 + 2 : client_id cohérent, nombre de localisations variable
#chaque client génère ses propres lignes, volume lié au statut_actif
locations_rows = []

for _, c in clients.iterrows():

    # Zone réseau assignée une fois par client (stable géographiquement)
    zone = np.random.choice(
        ["URBAIN", "SUBURBAIN", "RURAL"], p=[0.60, 0.30, 0.10]
    )
    cfg  = zones_config[zone]

    # ── Amélioration 2 : volume conditionné au statut ────────────────────────
    # Un client churné génère peu de points de localisation récents
    lam_loc = 6 if c.statut_actif else 2
    n_loc   = max(1, np.random.poisson(lam_loc))

    # ── Amélioration 6 : cohérence zone / plan tarifaire ────────────────────
    # Un client rural avec "Forfait Illimité" est peu probable
    # On le redescend doucement vers "Forfait Mobile (Mixte)"
    plan_effectif = c.plan_tarifaire
    if zone == "RURAL" and plan_effectif == "Forfait Illimité":
        plan_effectif = np.random.choice(
            ["Forfait Mobile (Mixte)", "Offre Classique"], p=[0.4, 0.6]
        )

    for _ in range(n_loc):

        # Coordonnées tirées dans les bornes de la zone
        lat = np.round(np.random.uniform(*cfg["lat"]), 6)
        lon = np.round(np.random.uniform(*cfg["lon"]), 6)

        # ── Amélioration 5 : qualité signal conditionnée à la zone ───────────
        #Excellent/Bon/Faible avec probabilités réalistes par zone
        qualite = np.random.choice(
            ["Excellent", "Bon", "Faible"], p=cfg["p_signal"]
        )

        # ── Amélioration 4 : timestamp de localisation ───────────────────────
        #date dans la période d'abonnement du client
        delta = (date_extraction - c.date_debut_abonnement).days
        if delta <= 0:
            delta = 1
        ts_loc = c.date_debut_abonnement + pd.Timedelta(
            days=int(np.random.randint(0, delta))
        )

        locations_rows.append({
            "client_id"           : c.client_id,
            "date_heure_connexion": ts_loc,
            "latitude_connexion"  : lat,
            "longitude_connexion" : lon,
            "zone_reseau"         : zone,
            "qualite_signal"      : qualite,
            "plan_effectif_zone"  : plan_effectif,   # plan cohérent avec la zone
        })

locations = pd.DataFrame(locations_rows)

# Tri chronologique
locations = locations.sort_values("date_heure_connexion").reset_index(drop=True)

# ── Vérification rapide ──────────────────────────────────────────────────────
print(f"Localisations générées        : {len(locations)}")
print(f"Localisations par client (moy): {len(locations)/n_clients:.1f}")
print(f"\nRépartition zones réseau :\n{locations['zone_reseau'].value_counts()}")
print(f"\nQualité signal par zone :")
print(locations.groupby("zone_reseau")["qualite_signal"].value_counts())

Localisations générées        : 1421
Localisations par client (moy): 4.7

Répartition zones réseau :
zone_reseau
URBAIN       831
SUBURBAIN    440
RURAL        150
Name: count, dtype: int64

Qualité signal par zone :
zone_reseau  qualite_signal
RURAL        Faible             80
             Bon                57
             Excellent          13
SUBURBAIN    Bon               222
             Excellent         123
             Faible             95
URBAIN       Excellent         500
             Bon               295
             Faible             36
Name: count, dtype: int64


Ce bloc simule les points de connexion géolocalisés de chaque client sur la période d'observation. Il a été entièrement restructuré pour garantir la cohérence géographique, relationnelle et temporelle des données :

Cohérence client_id : remplacement de np.random.choice (qui laissait certains clients sans localisation) par une boucle par client garantissant une couverture complète et contrôlée.

Volume variable lié au statut : le nombre de points de localisation suit une loi de Poisson conditionnée au statut du client (λ=6 si actif, λ=2 si churné), cohérent avec la logique des blocs CDR et sessions.

Cohérence géographique zone / coordonnées : chaque client se voit d'abord assigner une zone réseau (URBAIN, SUBURBAIN, RURAL), et ses coordonnées GPS sont ensuite tirées dans les bornes géographiques tunisiennes correspondant à cette zone — éliminant le cas absurde d'un point urbain en plein désert.

Qualité du signal conditionnée à la zone : variable ajoutée avec des probabilités différenciées par zone (signal excellent majoritaire en urbain, signal faible majoritaire en rural). Cette feature est un prédicteur de churn direct — une mauvaise couverture génère de l'insatisfaction.

Horodatage des localisations : chaque point est daté dans la fenêtre d'abonnement du client, permettant une jointure temporelle approximative avec la table sessions.

Cohérence zone / plan tarifaire : un client en zone rurale ayant initialement un "Forfait Illimité" est redirigé vers une offre plus modeste, reflétant la réalité de l'accessibilité réseau et commerciale dans les régions éloignées.


In [30]:
# ── Agrégation et construction du dataset final ──────────────────────────────


# ── Amélioration 1 : agrégation CDR enrichie ────────────────────────────────
#sommes + moyennes + comptages + récence (jours depuis dernier événement)
resume_usage = usage.groupby("client_id").agg(
    duree_appel_totale_sec = ("duree_appel_sec", "sum"),
    duree_appel_moyenne_sec = ("duree_appel_sec",lambda x: (x[x > 0].sum() / (x > 0).sum()) if (x > 0).any() else 0),
    nb_appels              = ("duree_appel_sec", lambda x: (x > 0).sum()),
    sms_total              = ("sms_compte",      "sum"),
    data_totale_mb         = ("data_mb",         "sum"),
    data_moyenne_mb         = ("data_mb",lambda x: (x[x > 0].sum() / (x > 0).sum()) if (x > 0).any() else 0),
    nb_evenements_data_cdr  = ("type_evenement",lambda x: (x == "donnee_mobile").sum()),
    nb_evenements_total    = ("type_evenement",  "count"),
    date_dernier_evenement = ("date_heure",      "max"),   # pour la récence
).reset_index()

# Récence : nombre de jours depuis le dernier événement CDR
resume_usage["recence_cdr_jours"] = (
    date_extraction - resume_usage["date_dernier_evenement"]
).dt.days
resume_usage = resume_usage.drop(columns=["date_dernier_evenement"])


# ── Amélioration 2 : agrégation sessions enrichie ───────────────────────────
#nombre de sessions, durée moyenne, récence, taux de cookies
resume_sessions = sessions.groupby("client_id").agg(
    nb_sessions              = ("duree_session_sec", "count"),
    duree_session_moyenne_sec= ("duree_session_sec", "mean"),
    duree_session_totale_sec = ("duree_session_sec", "sum"),
    taux_cookies             = ("cookies_acceptes",  "mean"),  # proportion [0,1]
    date_derniere_session    = ("date_heure_session", "max"),  # pour la récence
).reset_index()

# Récence : nombre de jours depuis la dernière session applicative
resume_sessions["recence_session_jours"] = (
    date_extraction - resume_sessions["date_derniere_session"]
).dt.days
resume_sessions = resume_sessions.drop(columns=["date_derniere_session"])


# ── Amélioration 3 : agrégation locations propre ────────────────────────────
#zone dominante via mode(), coordonnées moyennes, signal dominant
resume_locations = locations.groupby("client_id").agg(
    zone_reseau_principale = ("zone_reseau",         lambda x: x.mode()[0]),
    qualite_signal_dominante=("qualite_signal",      lambda x: x.mode()[0]),
    latitude_moyenne       = ("latitude_connexion",  "mean"),
    longitude_moyenne      = ("longitude_connexion", "mean"),
).reset_index()


# ── Amélioration 4 : suppression de clients.sample(frac=0.8) ────────────────
#tous les clients sont conservés — le churn est porté par la colonne 'churn' ( /1) encodée au bloc 1 à partir de statut_actif
dataset_final = clients \
    .merge(resume_usage,     on="client_id", how="left") \
    .merge(resume_sessions,  on="client_id", how="left") \
    .merge(resume_locations, on="client_id", how="left")


# ── Amélioration 5 : imputation des NaN post-merge ──────────────────────────
# Un client sans événements CDR a 0 appels/SMS/data — pas une valeur inconnue
# Un client sans session a 0 sessions — idem
colonnes_usage_zero = [
    "duree_appel_totale_sec", "duree_appel_moyenne_sec", "nb_appels",
    "sms_total", "data_totale_mb", "data_moyenne_mb",
    "nb_evenements_total", "recence_cdr_jours",
]
colonnes_sessions_zero = [
    "nb_sessions", "duree_session_moyenne_sec",
    "duree_session_totale_sec", "taux_cookies", "recence_session_jours",
]

dataset_final[colonnes_usage_zero]    = dataset_final[colonnes_usage_zero].fillna(0)
dataset_final[colonnes_sessions_zero] = dataset_final[colonnes_sessions_zero].fillna(0)

# ── Cas particulier : recence_cdr_jours ──────────────────────────────────────
# NaN = client sans AUCUN événement CDR → inactif depuis le début de son abonnement
# On impute avec tenure_jours (inactif depuis le premier jour) et non 0
# fillna(0) aurait donné recence=0 → churn=0 : contradiction totale
dataset_final["recence_cdr_jours"] = dataset_final["recence_cdr_jours"].fillna(dataset_final["tenure_jours"])

# ── Calcul de la variable cible churn ────────────────────────────────────────
# Règle métier scénario 1 (Churn Mobile Prépayé) :
# churn = 1 si aucune activité CDR depuis plus de 90 jours
# churn = 0 sinon
SEUIL_INACTIVITE = 90
dataset_final["churn"] = (
    dataset_final["recence_cdr_jours"] >= SEUIL_INACTIVITE
).astype(int)

# ── CORRECTION 5 : cohérence offre / consommation ────────────────────────────
# Offre Classique : pas d'accès internet → toutes les colonnes data à 0
mask_classique = dataset_final["plan_tarifaire"] == "Offre Classique"
dataset_final.loc[mask_classique, [
    "data_totale_mb", "data_moyenne_mb", "nb_evenements_data_cdr"
]] = 0

# Forfait Illimité : data uniquement → appels et SMS à 0
mask_illimite = dataset_final["plan_tarifaire"] == "Forfait Illimité"
dataset_final.loc[mask_illimite, [
    "nb_appels", "duree_appel_totale_sec", "duree_appel_moyenne_sec", "sms_total"
]] = 0

dataset_final["nb_evenements_total"] = (
    dataset_final["nb_appels"] +
    dataset_final["sms_total"] +
    dataset_final["nb_evenements_data_cdr"]
)

# ── Amélioration 6 : vérification qualité du dataset final ──────────────────
#audit minimal indispensable avant tout entraînement ML
print("=" * 55)
print(f"Dimensions du dataset final : {dataset_final.shape}")
print(f"\nDistribution variable cible (churn) :")
print(dataset_final["churn"].value_counts())
print(f"Taux de churn : {dataset_final['churn'].mean():.1%}")
print(f"Taux de churn (règle métier 90j) : {dataset_final['churn'].mean():.1%}")
print(f"Clients churné  : {dataset_final['churn'].sum()}")
print(f"Clients actifs  : {(dataset_final['churn'] == 0).sum()}")

Dimensions du dataset final : (300, 39)

Distribution variable cible (churn) :
churn
0    217
1     83
Name: count, dtype: int64
Taux de churn : 27.7%
Taux de churn (règle métier 90j) : 27.7%
Clients churné  : 83
Clients actifs  : 217


In [31]:
# ── Introduction de données manquantes (simulation d'imperfections réelles) ──


# ── Amélioration 1 : noms de colonnes cohérents avec le bloc 5 ───────────────
# Avant : 'data_mb' n'existe plus après agrégation → colonne fantôme créée
#noms alignés sur ceux produits par le groupby du bloc 5
# Note métier : NaN sur 'data_totale_mb' = client sans smartphone
#               ou sans option internet activée (Silent Churner potentiel)
col_a_salir = ["facture_moyenne_mensuelle", "data_totale_mb"]

for i, col in enumerate(col_a_salir):
    # ── Amélioration 2 : random_state fixe pour reproductibilité ─────────────
    #random_state unique par colonne (42+i) → résultats identiques
    idx = dataset_final.sample(frac=0.07, random_state=42 + i).index
    dataset_final.loc[idx, col] = np.nan


# ── Amélioration 3 : NaN satisfaction liés au statut (mécanisme MAR) ─────────
#MAR  — les clients inactifs (Silent Churners) répondent moins
#         aux enquêtes → probabilité de NaN plus élevée pour statut_actif=False
# idx_manquant stocke les indices des clients dont la satisfaction sera masquée
prob_nan_satisfaction = np.where(
    dataset_final["statut_actif"], 0.03, 0.15
)
masque_satisfaction = np.random.binomial(1, prob_nan_satisfaction).astype(bool)
idx_manquant = dataset_final[masque_satisfaction].index
dataset_final.loc[idx_manquant, "satisfaction_client"] = np.nan


# ── Amélioration 4 : deux types d'erreurs GPS ────────────────────────────────
#NaN pur (données non collectées) + (0.0, 0.0) (bug GPS classique)
#         Les deux cas doivent être détectés et traités différemment en nettoyage

# Cas 1 : valeur absente — GPS non disponible (5% des clients)
idx_gps_nan = dataset_final.sample(frac=0.05, random_state=50).index
dataset_final.loc[idx_gps_nan, ["latitude_moyenne", "longitude_moyenne"]] = np.nan

# Cas 2 : coordonnées aberrantes (0.0, 0.0) — bug de géolocalisation (2%)
idx_gps_aberrant = dataset_final.sample(frac=0.02, random_state=51).index
dataset_final.loc[idx_gps_aberrant, "latitude_moyenne"]  = 0.0
dataset_final.loc[idx_gps_aberrant, "longitude_moyenne"] = 0.0


# ── Amélioration 5 : colonnes indicatrices pour les NaN informatifs ──────────
# Certains NaN portent une information métier exploitable par le modèle ML
# 'data_totale_mb = NaN' → pas de smartphone ou option internet non activée
# 'satisfaction_client = NaN' → client non répondant (souvent Silent Churner)
dataset_final["data_manquante"]          = dataset_final["data_totale_mb"].isna().astype(int)
dataset_final["satisfaction_manquante"]  = dataset_final["satisfaction_client"].isna().astype(int)


# ── Amélioration 6 : audit cumulé post-salissage ─────────────────────────────
#vérification que le salissage est conforme et le dataset exploitable
print("=" * 55)
print("Audit des données manquantes post-salissage")
print("=" * 55)

nan_summary = dataset_final.isnull().sum()
nan_pct     = (nan_summary / len(dataset_final) * 100).round(2)
audit = nan_summary[nan_summary > 0].to_frame("nb_nan")
audit["pct"] = nan_pct[nan_summary > 0]
print(audit.sort_values("pct", ascending=False))

# Contrôle du cumul de NaN par ligne
nan_par_ligne = dataset_final.isnull().sum(axis=1)
print(f"\nClients sans aucun NaN      : {(nan_par_ligne == 0).sum()}")
print(f"Clients avec 1-2 NaN        : {((nan_par_ligne >= 1) & (nan_par_ligne <= 2)).sum()}")
print(f"Clients avec 3+ NaN (risqué): {(nan_par_ligne >= 3).sum()}")

# Vérification du taux de NaN GPS aberrants
gps_aberrant = ((dataset_final["latitude_moyenne"]  == 0.0) &
                (dataset_final["longitude_moyenne"] == 0.0)).sum()
print(f"\nCoordonnées GPS (0.0, 0.0)  : {gps_aberrant} clients")
print("=" * 55)

Audit des données manquantes post-salissage
                           nb_nan    pct
date_consentement             122  40.67
facture_moyenne_mensuelle      21   7.00
data_totale_mb                 21   7.00
satisfaction_client            18   6.00
latitude_moyenne               14   4.67
longitude_moyenne              14   4.67

Clients sans aucun NaN      : 134
Clients avec 1-2 NaN        : 159
Clients avec 3+ NaN (risqué): 7

Coordonnées GPS (0.0, 0.0)  : 6 clients


Ce bloc introduit des imperfections réalistes dans le dataset pour reproduire les conditions réelles d'un système d'information télécom. Trois types de données manquantes sont simulés, chacun avec une justification métier distincte.

Facture et consommation data (data_totale_mb) : 7% des valeurs sont masquées. Un NaN sur la consommation data peut indiquer qu'un client ne possède pas de smartphone, ou que l'option internet n'est pas activée sur son abonnement — ce qui constitue en soi une information sur son profil d'usage.

Satisfaction client : les NaN suivent un mécanisme MAR (Missing At Random) plutôt que MCAR (Missing Completely At Random) — les clients inactifs, qualifiés de Silent Churners (clients ayant toujours un contrat actif mais n'utilisant plus leur ligne), répondent statistiquement moins aux enquêtes de satisfaction. Le taux de NaN est donc de 15% pour les clients inactifs contre 3% pour les clients actifs. Ce mécanisme est important car il influence le choix de la stratégie d'imputation lors du prétraitement ML.

Coordonnées GPS : deux types d'erreurs sont distingués — les valeurs absentes (NaT, 5% des clients, correspondant à un refus de géolocalisation ou une absence de signal) et les coordonnées aberrantes (0.0, 0.0) (2% des clients), bug classique de géolocalisation mobile retournant les coordonnées par défaut au lieu d'une erreur. Ces deux cas requièrent des traitements différents lors du nettoyage.

Colonnes indicatrices : deux variables binaires (data_manquante, satisfaction_manquante) signalent explicitement les lignes concernées. Certains modèles comme XGBoost peuvent exploiter directement ces indicateurs comme features supplémentaires.

Note sur la reproductibilité : chaque sample() est ancré sur un random_state fixe et distinct, garantissant que le salissage produit exactement les mêmes indices à chaque exécution, indépendamment du seed global NumPy.

In [32]:
# ── Génération des réclamations clients ──────────────────────────────────────


# ── Amélioration 2 : table de correspondance satisfaction → λ ────────────────
# Avant : seuil binaire >3 / ≤3 — perd toute la granularité de l'échelle 1-5
# Après : λ graduel — plus la satisfaction est basse, plus les réclamations
#         sont fréquentes, avec une progression continue sur les 5 niveaux
lambda_par_satisfaction = {1: 4.0, 2: 3.0, 3: 2.0, 4: 1.0, 5: 0.3}


def generer_reclamations(s, actif):
    """
    Génère le nombre de réclamations selon la satisfaction et le statut.

    - Si satisfaction = NaN → NaN (client non répondant, profil inconnu)
    - λ graduel selon les 5 niveaux de satisfaction
    - Multiplicateur x1.5 si client inactif (churné) : plus de réclamations
      indépendamment de la satisfaction déclarée
    """
    # ── Amélioration 1 : NaN satisfaction → NaN réclamations ─────────────────
    # Avant : NaN > 3 = False silencieux → tous les NaN traitées comme λ=2
    # Après : NaN explicitement propagé — profil inconnu reste inconnu
    if pd.isna(s):
        return np.nan

    lam = lambda_par_satisfaction[int(s)]

    # ── Amélioration 3 : multiplicateur statut_actif ──────────────────────────
    # Un client churné réclame davantage, indépendamment de sa satisfaction
    if not actif:
        lam = min(lam * 1.5, 6.0)   # plafond à 6 pour rester réaliste

    return np.random.poisson(lam)


# Application ligne par ligne pour gérer les deux dimensions (satisfaction + statut)
dataset_final["nb_reclamations"] = [
    generer_reclamations(s, actif)
    for s, actif in zip(
        dataset_final["satisfaction_client"],
        dataset_final["statut_actif"]
    )
]


# ── Amélioration 4 : random_state fixe ───────────────────────────────────────
# Avant : sample() sans random_state → non reproductible
# Après : random_state=60 distinct des blocs précédents (50, 51, 42+i...)
# Note métier : NaN ≠ 0 — un client avec nb_reclamations=NaN n'a jamais
#               contacté le support (Silent Churner potentiel)
#               un client avec nb_reclamations=0 a contacté le support
#               sans avoir eu de problème à signaler
idx_no_contact = dataset_final.sample(frac=0.10, random_state=60).index
dataset_final.loc[idx_no_contact, "nb_reclamations"] = np.nan


# ── Amélioration 5 : colonne indicatrice ─────────────────────────────────────
# NaN sur nb_reclamations = information métier exploitable par le modèle ML
# (jamais contacté le support ≠ zéro réclamation)
dataset_final["reclamation_manquante"] = (
    dataset_final["nb_reclamations"].isna().astype(int)
)


# ── Amélioration 6 : audit final global ──────────────────────────────────────
print("=" * 60)
print("AUDIT FINAL — Dataset prêt pour le pipeline ML")
print("=" * 60)

print(f"\nDimensions           : {dataset_final.shape}")
print(f"Clients              : {len(dataset_final)}")
print(f"Features disponibles : {dataset_final.shape[1]}")

print(f"\nVariable cible (churn) :")
vc = dataset_final["churn"].value_counts()
print(f"  Actifs   (0) : {vc[0]} ({vc[0]/len(dataset_final):.1%})")
print(f"  Churnés  (1) : {vc[1]} ({vc[1]/len(dataset_final):.1%})")

print(f"\nNaN par colonne (colonnes concernées uniquement) :")
nan_final = dataset_final.isnull().sum()
nan_final = nan_final[nan_final > 0].sort_values(ascending=False)
for col, n in nan_final.items():
    print(f"  {col:<35} {n:>4} ({n/len(dataset_final):.1%})")

print(f"\nColonnes indicatrices de NaN :")
for col in ["data_manquante", "satisfaction_manquante", "reclamation_manquante"]:
    n = dataset_final[col].sum()
    print(f"  {col:<35} {n:>4} clients concernés")

print(f"\nAperçu des features pour le modèle ML :")
features_ml = [
    "tenure_mois", "facture_moyenne_mensuelle", "plan_tarifaire",
    "duree_appel_totale_sec", "nb_appels", "sms_total",
    "data_totale_mb", "recence_cdr_jours",
    "nb_sessions", "recence_session_jours",
    "zone_reseau_principale", "qualite_signal_dominante",
    "satisfaction_client", "nb_reclamations", "churn"
]
print(dataset_final[features_ml].describe().round(2))
print("=" * 60)

AUDIT FINAL — Dataset prêt pour le pipeline ML

Dimensions           : (300, 43)
Clients              : 300
Features disponibles : 43

Variable cible (churn) :
  Actifs   (0) : 217 (72.3%)
  Churnés  (1) : 83 (27.7%)

NaN par colonne (colonnes concernées uniquement) :
  date_consentement                    122 (40.7%)
  nb_reclamations                       48 (16.0%)
  facture_moyenne_mensuelle             21 (7.0%)
  data_totale_mb                        21 (7.0%)
  satisfaction_client                   18 (6.0%)
  latitude_moyenne                      14 (4.7%)
  longitude_moyenne                     14 (4.7%)

Colonnes indicatrices de NaN :
  data_manquante                        21 clients concernés
  satisfaction_manquante                18 clients concernés
  reclamation_manquante                 48 clients concernés

Aperçu des features pour le modèle ML :
       tenure_mois  facture_moyenne_mensuelle  duree_appel_totale_sec  \
count       300.00                     279.00     

In [33]:
# ══════════════════════════════════════════════════════════════════════════════
# FEATURE ENGINEERING — Partie 1 : Tendance de consommation data (mois M vs M-1)
# ══════════════════════════════════════════════════════════════════════════════

# ── Étape 1 : isoler les événements data depuis le CDR brut ──────────────────
# On travaille sur `usage` AVANT agrégation car c'est là que date_heure existe
cdr_data = usage[usage["type_evenement"] == "donnee_mobile"].copy()

# ── Étape 2 : extraire l'année-mois de chaque événement ─────────────────────
cdr_data["annee_mois"] = cdr_data["date_heure"].dt.to_period("M")

# ── Étape 3 : sommer la data par client et par mois ─────────────────────────
data_par_mois = (
    cdr_data
    .groupby(["client_id", "annee_mois"])["data_mb"]
    .sum()
    .reset_index()
    .sort_values(["client_id", "annee_mois"])
)

# ── Étape 4 : extraire M (dernier mois) et M-1 (avant-dernier) par client ───
def extraire_M_et_M1(group):
    """
    Pour un client donné, retourne sa consommation data du dernier mois (M)
    et du mois précédent (M-1).
    - 0 mois disponible  → M=0, M-1=0  (client sans aucun événement data)
    - 1 mois disponible  → M=valeur, M-1=0  (pas d'historique antérieur)
    - 2+ mois disponibles → M=dernier, M-1=avant-dernier
    """
    mois_tries = group.sort_values("annee_mois")
    if len(mois_tries) == 0:
        return pd.Series({"data_mois_M": 0.0, "data_mois_M1": 0.0})
    elif len(mois_tries) == 1:
        return pd.Series({"data_mois_M": mois_tries["data_mb"].iloc[-1],
                          "data_mois_M1": 0.0})
    else:
        return pd.Series({"data_mois_M":  mois_tries["data_mb"].iloc[-1],
                          "data_mois_M1": mois_tries["data_mb"].iloc[-2]})

data_trend = (
    data_par_mois
    .groupby("client_id")
    .apply(extraire_M_et_M1)
    .reset_index()
)

# ── Étape 5 : merger dans dataset_final ─────────────────────────────────────
dataset_final = dataset_final.merge(data_trend, on="client_id", how="left")

# Les clients sans aucun événement data (Offre Classique, inactifs) → 0
dataset_final[["data_mois_M", "data_mois_M1"]] = (
    dataset_final[["data_mois_M", "data_mois_M1"]].fillna(0.0)
)

# ── Étape 6 corrigée : calcul de la tendance sans EPSILON ───────────────────

def calculer_tendance(row):
    M  = row["data_mois_M"]
    M1 = row["data_mois_M1"]
    
    if M == 0 and M1 == 0:
        return 0.0          # aucune activité data sur les 2 mois
    elif M1 == 0 and M > 0:
        return np.nan       # nouveau consommateur data ce mois-ci : non calculable
    else:
        return round((M - M1) / M1 * 100, 2)

dataset_final["tendance_data_pct"] = dataset_final.apply(calculer_tendance, axis=1)

# ── Étape 7 : cohérence métier — Offre Classique ─────────────────────────────
mask_classique = dataset_final["plan_tarifaire"] == "Offre Classique"
dataset_final.loc[mask_classique, 
    ["data_mois_M", "data_mois_M1", "tendance_data_pct"]] = 0.0

# ── Vérification ──────────────────────────────────────────────────────────────
print(dataset_final["tendance_data_pct"].describe().round(2))
print(f"\nNaN (nouveau consommateur data) : {dataset_final['tendance_data_pct'].isna().sum()}")
print(f"Valeurs > 1000 : {(dataset_final['tendance_data_pct'] > 1000).sum()}")

count     290.00
mean       42.99
std       231.59
min       -93.68
25%         0.00
50%         0.00
75%         0.00
max      2544.04
Name: tendance_data_pct, dtype: float64

NaN (nouveau consommateur data) : 10
Valeurs > 1000 : 4


C:\Users\Paul Adrien\AppData\Local\Temp\ipykernel_53108\4072945404.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(extraire_M_et_M1)


In [34]:
# ══════════════════════════════════════════════════════════════════════════════
# FEATURE ENGINEERING — Partie 2 : Ratios d'usage
# ══════════════════════════════════════════════════════════════════════════════

# ── Feature 2 : ratio_sms_appels ─────────────────────────────────────────────
# Mesure la préférence SMS vs Voix
# NaN si nb_appels = 0 mais sms_total > 0 (profil SMS-only, non calculable)
# 0.0 si les deux sont nuls (client inactif sur voix/SMS)

def calc_ratio_sms_appels(row):
    sms = row["sms_total"]
    app = row["nb_appels"]
    if app == 0 and sms == 0:
        return 0.0
    elif app == 0 and sms > 0:
        return np.nan
    else:
        return round(sms / app, 4)

dataset_final["ratio_sms_appels"] = dataset_final.apply(
    calc_ratio_sms_appels, axis=1
)


# ── Feature 3 : ratio_data_voix ──────────────────────────────────────────────
# Mesure l'orientation "Digital Native" : data (MB) par seconde de voix
# NaN si durée voix = 0 mais data > 0 (profil data-only, ex: Forfait Illimité)
# 0.0 si les deux sont nuls (client inactif ou Offre Classique)

def calc_ratio_data_voix(row):
    data  = row["data_totale_mb"]
    duree = row["duree_appel_totale_sec"]
    if pd.isna(data):
        data = 0.0
    if duree == 0 and data == 0:
        return 0.0
    elif duree == 0 and data > 0:
        return np.nan
    else:
        return round(data / duree, 6)

dataset_final["ratio_data_voix"] = dataset_final.apply(
    calc_ratio_data_voix, axis=1
)


# ── Vérification ──────────────────────────────────────────────────────────────
print("=== ratio_sms_appels ===")
print(dataset_final["ratio_sms_appels"].describe().round(3))
print(f"NaN (profil SMS-only) : {dataset_final['ratio_sms_appels'].isna().sum()}")

print("\n=== ratio_data_voix ===")
print(dataset_final["ratio_data_voix"].describe().round(6))
print(f"NaN (profil data-only) : {dataset_final['ratio_data_voix'].isna().sum()}")

print("\n=== Aperçu par plan tarifaire ===")
print(dataset_final.groupby("plan_tarifaire")[
    ["ratio_sms_appels", "ratio_data_voix"]
].mean().round(3))

=== ratio_sms_appels ===
count    289.000
mean       0.773
std        0.649
min        0.000
25%        0.333
50%        0.667
75%        1.000
max        3.500
Name: ratio_sms_appels, dtype: float64
NaN (profil SMS-only) : 11

=== ratio_data_voix ===
count    273.000000
mean       0.064047
std        0.642069
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max       10.017500
Name: ratio_data_voix, dtype: float64
NaN (profil data-only) : 27

=== Aperçu par plan tarifaire ===
                        ratio_sms_appels  ratio_data_voix
plan_tarifaire                                           
Forfait Illimité                   0.000            0.000
Forfait Mobile (Mixte)             0.885            0.278
Offre Classique                    0.842            0.000


In [35]:
# ── Encodage ordinal des deux variables ──────────────────────────────────────
score_zone = {
    "URBAIN"    : 3,
    "SUBURBAIN" : 2,
    "RURAL"     : 1,
}
score_signal = {
    "Excellent" : 3,
    "Bon"       : 2,
    "Faible"    : 1,
}

# ── Score = produit des deux encodages ───────────────────────────────────────
dataset_final["score_qualite_zone"] = dataset_final.apply(
    lambda row: (
        score_zone.get(row["zone_reseau_principale"], np.nan) *
        score_signal.get(row["qualite_signal_dominante"], np.nan)
    ),
    axis=1
)

# ── Vérification ──────────────────────────────────────────────────────────────
print("=== score_qualite_zone ===")
print(dataset_final["score_qualite_zone"].describe().round(2))
print(f"\nNaN : {dataset_final['score_qualite_zone'].isna().sum()}")

print("\n=== Répartition des scores ===")
print(dataset_final["score_qualite_zone"].value_counts().sort_index())

print("\n=== Score moyen par zone ===")
print(dataset_final.groupby("zone_reseau_principale")["score_qualite_zone"].mean().round(2))

print("\n=== Aperçu ===")
print(dataset_final[["client_id", "zone_reseau_principale",
                      "qualite_signal_dominante",
                      "score_qualite_zone"]].head(15).to_string())

=== score_qualite_zone ===
count    300.00
mean       6.11
std        2.58
min        1.00
25%        4.00
50%        6.00
75%        9.00
max        9.00
Name: score_qualite_zone, dtype: float64

NaN : 0

=== Répartition des scores ===
score_qualite_zone
1     16
2     22
3      1
4     62
6     90
9    109
Name: count, dtype: int64

=== Score moyen par zone ===
zone_reseau_principale
RURAL        1.53
SUBURBAIN    4.41
URBAIN       7.82
Name: score_qualite_zone, dtype: float64

=== Aperçu ===
    client_id zone_reseau_principale qualite_signal_dominante  score_qualite_zone
0           1                  RURAL                   Faible                   1
1           2                 URBAIN                      Bon                   6
2           3              SUBURBAIN                      Bon                   4
3           4                 URBAIN                      Bon                   6
4           5                 URBAIN                      Bon                   6
5       

In [36]:
# ══════════════════════════════════════════════════════════════════════════════
# FEATURE ENGINEERING — Partie 4 : Score de Frustration
# ══════════════════════════════════════════════════════════════════════════════

# ── Formule : nb_reclamations × (6 - satisfaction_client) ────────────────────
# Principe 1 : inversion de l'échelle de Likert 1→5 en poids 5→1
#              le +1 (soit le 6) garantit un poids minimum de 1, jamais 0
# Principe 2 : multiplication = effet d'amplification mutuelle
#              une réclamation d'un client insatisfait pèse plus lourd
# Cas NaN    : si satisfaction OU réclamations inconnues → score inconnu (NaN)

dataset_final["score_frustration"] = (
    dataset_final["nb_reclamations"] *
    (6 - dataset_final["satisfaction_client"])
)

# ── Vérification ──────────────────────────────────────────────────────────────
print("=== score_frustration ===")
print(dataset_final["score_frustration"].describe().round(2))
print(f"\nNaN : {dataset_final['score_frustration'].isna().sum()}")

print("\n=== Score moyen par niveau de churn ===")
print(dataset_final.groupby("churn")["score_frustration"].mean().round(2))

print("\n=== Aperçu ===")
print(dataset_final[["client_id", "nb_reclamations",
                      "satisfaction_client",
                      "score_frustration", "churn"]].head(15).to_string())

=== score_frustration ===
count    252.00
mean      13.10
std       12.21
min        0.00
25%        3.00
50%       10.00
75%       20.00
max       65.00
Name: score_frustration, dtype: float64

NaN : 48

=== Score moyen par niveau de churn ===
churn
0    11.11
1    18.61
Name: score_frustration, dtype: float64

=== Aperçu ===
    client_id  nb_reclamations  satisfaction_client  score_frustration  churn
0           1              0.0                  5.0                0.0      0
1           2              NaN                  1.0                NaN      0
2           3              NaN                  NaN                NaN      0
3           4              3.0                  2.0               12.0      0
4           5              6.0                  1.0               30.0      0
5           6              3.0                  3.0                9.0      0
6           7              NaN                  NaN                NaN      1
7           8              7.0                 

In [37]:
# ── Export du dataset final ───────────────────────────────────────────────────
from pathlib import Path
import datetime

# ── Amélioration 1 : chemin portable via pathlib ─────────────────────────────
# Avant : chemin relatif fragile avec ../../../../ — casse si le notebook bouge
# Après : chemin construit depuis la racine du projet, dossier créé si absent
output_dir = Path(r"C:\Users\Paul Adrien")
output_dir.mkdir(parents=True, exist_ok=True)   # crée le dossier si nécessaire

output_path = output_dir / "dataset_pfe_brut2.csv"
dataset_final.to_csv(output_path, index=False, encoding="utf-8-sig")
# encoding="utf-8-sig" : assure la lisibilité des caractères accentués
#                        dans Excel et les outils tunisiens


# ── Amélioration 2 : vérification de l'export ────────────────────────────────
# Avant : aucun retour — on ne sait pas si le fichier a été écrit correctement
# Après : confirmation explicite avec taille du fichier
assert output_path.exists(), f"Erreur : le fichier {output_path} n'a pas été créé."
taille_ko = output_path.stat().st_size / 1024
print(f"✓ Export réussi : {output_path}")
print(f"  Taille        : {taille_ko:.1f} Ko")
print(f"  Lignes        : {len(dataset_final)}")
print(f"  Colonnes      : {len(dataset_final.columns)}")


# ── Amélioration 3 : fichier de métadonnées ───────────────────────────────────
# Trace les paramètres de simulation pour la reproductibilité et le rapport PFE
meta_path = output_dir / "dataset_pfe_metadata.txt"

with open(meta_path, "w", encoding="utf-8") as f:
    f.write("=" * 55 + "\n")
    f.write("MÉTADONNÉES — Dataset synthétique PFE Tunisie Télécom\n")
    f.write("=" * 55 + "\n\n")
    f.write(f"Date de génération   : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
    f.write(f"Seed aléatoire       : 42\n")
    f.write(f"Nombre de clients    : {len(dataset_final)}\n")
    f.write(f"Nombre de features   : {len(dataset_final.columns)}\n")
    f.write(f"Période simulée      : 2020-01-01 → 2025-01-01\n")
    f.write(f"Taux de churn        : {dataset_final['churn'].mean():.1%}\n\n")
    f.write("Colonnes du dataset  :\n")
    for col in dataset_final.columns:
        nan_pct = dataset_final[col].isna().mean() * 100
        f.write(f"  - {col:<40} (NaN : {nan_pct:.1f}%)\n")
    f.write("\n" + "=" * 55 + "\n")
    f.write("Tables sources utilisées :\n")
    f.write(f"  - clients   : {n_clients} lignes\n")
    f.write(f"  - usage     : {len(usage)} événements CDR\n")
    f.write(f"  - sessions  : {len(sessions)} sessions internet\n")
    f.write(f"  - locations : {len(locations)} points de localisation\n")

print(f"✓ Métadonnées       : {meta_path}")

✓ Export réussi : C:\Users\Paul Adrien\dataset_pfe_brut2.csv
  Taille        : 104.2 Ko
  Lignes        : 300
  Colonnes      : 50
✓ Métadonnées       : C:\Users\Paul Adrien\dataset_pfe_metadata.txt


In [38]:
# ── Aperçu aléatoire du dataset final ────────────────────────────────────────

# ── Amélioration 1 : random_state pour un aperçu reproductible ───────────────
# Avant : 10 lignes différentes à chaque exécution
# Après : même échantillon à chaque run → cohérence dans le rapport PFE
print("Aperçu de 10 lignes aléatoires — features analytiques\n")

# ── Amélioration 2 : colonnes ciblées sur les features ML ────────────────────
# Avant : toutes les colonnes affichées, y compris données administratives
# Après : sélection des features directement utiles au modèle churn
features_apercu = [
    # Profil client
    "client_id", "tenure_mois", "plan_tarifaire", "type_abonnement",
    "facture_moyenne_mensuelle", "moyen_paiement",
    # Usage télécom
    "nb_appels", "duree_appel_totale_sec", "sms_total",
    "data_totale_mb", "recence_cdr_jours",
    # Engagement digital
    "nb_sessions", "recence_session_jours", "taux_cookies",
    # Localisation
    "zone_reseau_principale", "qualite_signal_dominante",
    # Satisfaction & support
    "satisfaction_client", "nb_reclamations",
    # Indicateurs NaN
    "data_manquante", "satisfaction_manquante", "reclamation_manquante",
    # Variable cible
    "churn",
]

apercu = dataset_final[features_apercu].sample(10, random_state=42)
display(apercu)


# ── Amélioration 3 : assertions de cohérence métier ─────────────────────────
# Vérifie que les règles métier fondamentales sont respectées sur tout le dataset
print("\nVérifications de cohérence métier :")

# Règle 1 : un opt-out ne peut exister sans consentement préalable
assert not (
    (dataset_final["optout_marketing"] == True) &
    (dataset_final["consentement_marketing"] == False)
).any(), "❌ Des clients ont un opt-out sans consentement préalable."
print("  ✓ Cohérence optout / consentement respectée")

# Règle 2 : tenure toujours positive
assert (dataset_final["tenure_jours"] > 0).all(), \
    "❌ Certains clients ont une tenure nulle ou négative."
print("  ✓ Tenure strictement positive pour tous les clients")

# Règle 3 : churn aligné avec statut_actif
assert (
    dataset_final.loc[
        dataset_final["churn"] == 1, "recence_cdr_jours"
    ] >= SEUIL_INACTIVITE
).all(), "❌ Des clients marqués churn=1 ont une récence < 90 jours."
print("  ✓ Tous les clients churné ont bien une récence CDR ≥ 90 jours")

# Règle 4 : aucune coordonnée hors des frontières tunisiennes (hors aberrants 0.0)
coords_valides = dataset_final.dropna(subset=["latitude_moyenne", "longitude_moyenne"])
coords_valides = coords_valides[
    (coords_valides["latitude_moyenne"]  != 0.0) &
    (coords_valides["longitude_moyenne"] != 0.0)
]
assert coords_valides["latitude_moyenne"].between(30.0, 37.2).all() and \
       coords_valides["longitude_moyenne"].between(7.5, 11.6).all(), \
    "❌ Des coordonnées GPS sont hors des frontières tunisiennes."
print("  ✓ Coordonnées GPS dans les bornes tunisiennes")

print("\n✓ Toutes les vérifications passées — dataset prêt pour CRISP-DM Phase 3")

Aperçu de 10 lignes aléatoires — features analytiques



,client_id,tenure_mois,plan_tarifaire,type_abonnement,facture_moyenne_mensuelle,moyen_paiement,nb_appels,duree_appel_totale_sec,sms_total,data_totale_mb,...,recence_session_jours,taux_cookies,zone_reseau_principale,qualite_signal_dominante,satisfaction_client,nb_reclamations,data_manquante,satisfaction_manquante,reclamation_manquante,churn
203,204,38,Offre Classique,Offre Prépayée,NaN,ticket_recharge,3.0,340.0,1.0,NaN,...,227,0.500000,URBAIN,Excellent,2.0,5.0,1,0,0,0
266,267,59,Offre Classique,Offre Prépayée,NaN,ticket_recharge,4.0,538.0,7.0,0.00,...,292,0.700000,URBAIN,Bon,1.0,NaN,0,0,1,1
152,153,35,Forfait Illimité,Offre Prépayée,NaN,ticket_recharge,0.0,0.0,0.0,494.62,...,93,0.800000,RURAL,Bon,2.0,3.0,0,0,0,0
9,10,19,Offre Classique,Offre Prépayée,NaN,ticket_recharge,6.0,514.0,9.0,0.00,...,135,0.333333,URBAIN,Excellent,2.0,4.0,0,0,0,0
233,234,14,Offre Classique,Offre Prépayée,NaN,ticket_recharge,4.0,935.0,1.0,0.00,...,261,1.000000,URBAIN,Bon,NaN,NaN,0,1,1,0
226,227,55,Offre Classique,Offre Prépayée,NaN,prelevement_bancaire,5.0,1039.0,5.0,0.00,...,57,0.875000,SUBURBAIN,Bon,2.0,1.0,0,0,0,1
196,197,13,Offre Classique,Offre Prépayée,NaN,ticket_recharge,5.0,946.0,8.0,0.00,...,28,0.272727,URBAIN,Bon,5.0,0.0,0,0,0,0
109,110,47,Offre Classique,Offre Prépayée,NaN,ticket_recharge,1.0,35.0,1.0,0.00,...,257,1.000000,URBAIN,Bon,NaN,NaN,0,1,1,0
5,6,24,Forfait Mobile (Mixte),Offre Prépayée,NaN,ticket_recharge,10.0,1409.0,12.0,109.73,...,123,0.700000,URBAIN,Excellent,3.0,3.0,0,0,0,0
175,176,17,Forfait Mobile (Mixte),Offre Prépayée,NaN,prelevement_bancaire,12.0,2738.0,3.0,62.23,...,146,0.600000,URBAIN,Bon,2.0,3.0,0,0,0,0



Vérifications de cohérence métier :
  ✓ Cohérence optout / consentement respectée
  ✓ Tenure strictement positive pour tous les clients
  ✓ Tous les clients churné ont bien une récence CDR ≥ 90 jours
  ✓ Coordonnées GPS dans les bornes tunisiennes

✓ Toutes les vérifications passées — dataset prêt pour CRISP-DM Phase 3


In [39]:
dataset_final.to_csv(r"C:\Users\Paul Adrien\dataset_pfe_brut.csv", index=False, encoding="utf-8-sig")
print("✓ Fichier sauvegardé dans le dossier du notebook")

✓ Fichier sauvegardé dans le dossier du notebook


In [40]:
# ── Observation 1 : répartition des types d'abonnement ──────────────────────
# → Quel scénario est le plus représenté dans ta base clients ?
print("=" * 50)
print("1. Répartition des types d'abonnement")
print("=" * 50)
print(dataset_final["type_abonnement"].value_counts())
print(dataset_final["type_abonnement"].value_counts(normalize=True).map("{:.1%}".format))

# ── Observation 2 : distribution de la récence CDR ──────────────────────────
# → Est-ce qu'on peut appliquer une règle d'inactivité (ex: 90 jours) ?
# → Combien de clients seraient churné selon cette règle ?
print("\n" + "=" * 50)
print("2. Distribution de la récence CDR (jours depuis dernier événement)")
print("=" * 50)
print(dataset_final["recence_cdr_jours"].describe().round(1))

seuil_inactivite = 90
clients_inactifs_90j = (dataset_final["recence_cdr_jours"] >= seuil_inactivite).sum()
print(f"\nClients sans activité depuis +{seuil_inactivite} jours : "
      f"{clients_inactifs_90j} ({clients_inactifs_90j/len(dataset_final):.1%})")

# ── Observation 3 : variabilité de la consommation data ─────────────────────
# → La data est-elle assez variable pour détecter une "baisse drastique" ?
print("\n" + "=" * 50)
print("3. Distribution de la consommation data (MB)")
print("=" * 50)
print(dataset_final["data_totale_mb"].describe().round(2))
print(f"\nClients sans aucune consommation data (NaN ou 0) : "
      f"{(dataset_final['data_totale_mb'].fillna(0) == 0).sum()}")
print(f"Indicateur data_manquante = 1 : "
      f"{dataset_final['data_manquante'].sum()} clients")

# ── Observation 4 : croisement type_abonnement × churn actuel ───────────────
# → Le churn est-il concentré sur un type d'abonnement particulier ?
print("\n" + "=" * 50)
print("4. Taux de churn par type d'abonnement")
print("=" * 50)
print(dataset_final.groupby("type_abonnement")["churn"].mean().map("{:.1%}".format))

# ── Observation 5 : croisement plan_tarifaire × recence_cdr_jours ───────────
# → L'inactivité est-elle liée au plan tarifaire ?
print("\n" + "=" * 50)
print("5. Récence moyenne par plan tarifaire")
print("=" * 50)
print(dataset_final.groupby("plan_tarifaire")["recence_cdr_jours"].mean().round(1))

1. Répartition des types d'abonnement
type_abonnement
Offre Prépayée     248
Offre à Facture     52
Name: count, dtype: int64
type_abonnement
Offre Prépayée     82.7%
Offre à Facture    17.3%
Name: proportion, dtype: object

2. Distribution de la récence CDR (jours depuis dernier événement)
count     300.0
mean       97.4
std       171.3
min         0.0
25%        13.0
50%        38.0
75%       101.2
max      1640.0
Name: recence_cdr_jours, dtype: float64

Clients sans activité depuis +90 jours : 83 (27.7%)

3. Distribution de la consommation data (MB)
count    279.00
mean      49.39
std      127.05
min        0.00
25%        0.00
50%        0.00
75%       19.26
max      684.31
Name: data_totale_mb, dtype: float64

Clients sans aucune consommation data (NaN ou 0) : 215
Indicateur data_manquante = 1 : 21 clients

4. Taux de churn par type d'abonnement
type_abonnement
Offre Prépayée     25.8%
Offre à Facture    36.5%
Name: churn, dtype: object

5. Récence moyenne par plan tarifaire
plan_

In [85]:
import pandas as pd

df = pd.read_csv(r"C:/Users/Paul Adrien/Desktop/dataset_pfe_brut.csv")

# Colonnes admin inutiles pour SDV
cols_admin = [
    "nom_client", "adresse_email", "num_tel_mobile",
    "adresse_physique", "identifiant_national"
]

# Colonnes dérivées / indicateurs inutiles pour SDV
cols_a_exclure = [
    "tenure_mois", "nb_evenements_total",
    "data_manquante", "satisfaction_manquante", "reclamation_manquante",
    "data_mois_M", "data_mois_M1",
]

df_sdv = df.drop(columns=cols_admin + cols_a_exclure)
print(f"Shape pour SDV : {df_sdv.shape}")

Shape pour SDV : (300, 38)


In [86]:
from sdv.metadata import Metadata

metadata = Metadata()
metadata.detect_table_from_dataframe(
    table_name="clients_tt",
    data=df_sdv
)

# Corrections des types mal détectés
metadata.update_column(table_name="clients_tt",
    column_name="latitude_moyenne",  sdtype="numerical")
metadata.update_column(table_name="clients_tt",
    column_name="longitude_moyenne", sdtype="numerical")
metadata.update_column(table_name="clients_tt",
    column_name="score_qualite_zone", sdtype="numerical")
metadata.update_column(table_name="clients_tt",
    column_name="churn", sdtype="numerical")

print("Métadonnées prêtes.")

Métadonnées prêtes.


In [93]:
from sdv.single_table import GaussianCopulaSynthesizer

model = GaussianCopulaSynthesizer(
    metadata,
    numerical_distributions={
        "data_totale_mb"         : "truncnorm",
        "data_moyenne_mb"        : "truncnorm",
        "duree_appel_totale_sec" : "truncnorm",
        "score_frustration"      : "truncnorm",
    }
)
model.fit(df_sdv)
print("Modèle re-entraîné avec distributions corrigées.")

Modèle re-entraîné avec distributions corrigées.


In [94]:
model.fit(df_sdv)
print("Modèle entraîné.")

Modèle entraîné.


In [95]:
synthetic_df = model.sample(num_rows=300)
print(f"Shape : {synthetic_df.shape}")
synthetic_df.head()

Shape : (300, 38)


,client_id,genre_client,date_naissance,type_abonnement,plan_tarifaire,date_debut_abonnement,statut_actif,moyen_paiement,facture_moyenne_mensuelle,satisfaction_client,...,qualite_signal_dominante,latitude_moyenne,longitude_moyenne,churn,nb_reclamations,tendance_data_pct,ratio_sms_appels,ratio_data_voix,score_qualite_zone,score_frustration
0,1359044,Homme,1992-10-04,Offre Prépayée,Offre Classique,2022-07-13,False,especes,25.11,1.0,...,Excellent,36.987491,10.682401,0,5.0,-52.64,0.0751,0.341945,5,21.0
1,15694197,Femme,1954-07-15,Offre Prépayée,Offre Classique,2021-03-28,True,ticket_recharge,NaN,2.0,...,Excellent,36.987491,10.682401,1,9.0,-26.64,0.0065,0.032848,8,NaN
2,11190970,Homme,1981-01-26,Offre Prépayée,Offre Classique,2023-01-05,True,especes,49.58,3.0,...,Faible,36.617104,10.165643,1,2.0,-47.79,1.4162,0.003377,8,NaN
3,8611155,Femme,1945-07-11,Offre Prépayée,Offre Classique,2020-06-29,True,ticket_recharge,NaN,1.0,...,Excellent,25.364151,NaN,1,4.0,-86.34,0.5318,0.006393,6,23.0
4,1088135,Homme,1986-08-10,Offre à Facture,Offre Classique,2023-01-31,True,prelevement_bancaire,91.73,2.0,...,Excellent,36.496336,10.011242,1,9.0,-18.02,0.0954,0.025551,9,30.0


In [96]:
print("=== ORIGINAL ===")
print(f"Churn : {df_sdv['churn'].mean():.1%}")
print(f"Data moyenne : {df_sdv['data_totale_mb'].mean():.1f} MB")
print(f"Récence moyenne : {df_sdv['recence_cdr_jours'].mean():.1f} jours")
print(f"Plan tarifaire :\n{df_sdv['plan_tarifaire'].value_counts(normalize=True).round(2)}")

print("\n=== SYNTHÉTIQUE ===")
print(f"Churn : {synthetic_df['churn'].mean():.1%}")
print(f"Data moyenne : {synthetic_df['data_totale_mb'].mean():.1f} MB")
print(f"Récence moyenne : {synthetic_df['recence_cdr_jours'].mean():.1f} jours")
print(f"Plan tarifaire :\n{synthetic_df['plan_tarifaire'].value_counts(normalize=True).round(2)}")

=== ORIGINAL ===
Churn : 27.3%
Data moyenne : 49.4 MB
Récence moyenne : 109.1 jours
Plan tarifaire :
plan_tarifaire
Offre Classique           0.70
Forfait Mobile (Mixte)    0.21
Forfait Illimité          0.09
Name: proportion, dtype: float64

=== SYNTHÉTIQUE ===
Churn : 26.7%
Data moyenne : 100.1 MB
Récence moyenne : 108.2 jours
Plan tarifaire :
plan_tarifaire
Offre Classique           0.73
Forfait Mobile (Mixte)    0.20
Forfait Illimité          0.07
Name: proportion, dtype: float64


In [92]:
# Ces valeurs ne doivent pas apparaître dans les données synthétiques
problemes = []

# Churn doit être 0 ou 1 uniquement
valeurs_churn = synthetic_df['churn'].round().astype(int).unique()
if not set(valeurs_churn).issubset({0, 1}):
    problemes.append(f"Churn contient des valeurs hors 0/1 : {valeurs_churn}")

# Pas de valeurs négatives sur les colonnes de volume
for col in ["nb_appels", "sms_total", "data_totale_mb", "nb_sessions"]:
    n_neg = (synthetic_df[col] < 0).sum()
    if n_neg > 0:
        problemes.append(f"{col} : {n_neg} valeurs négatives")

# Récence ne doit pas être négative
n_neg_rec = (synthetic_df['recence_cdr_jours'] < 0).sum()
if n_neg_rec > 0:
    problemes.append(f"recence_cdr_jours : {n_neg_rec} valeurs négatives")

if problemes:
    print(" Problèmes détectés :")
    for p in problemes: print(f"  - {p}")
else:
    print("Aucun problème critique détecté")

Aucun problème critique détecté


In [98]:
# Correction post-génération — recalibration de data_totale_mb
import numpy as np

# Facteur de correction = ratio original / synthétique
facteur = df_sdv['data_totale_mb'].mean() / synthetic_df['data_totale_mb'].mean()
print(f"Facteur de correction : {facteur:.3f}")

# Application du facteur + clip pour éviter les valeurs négatives
synthetic_df['data_totale_mb'] = (
    synthetic_df['data_totale_mb'] * facteur
).clip(lower=0)

# Même correction sur data_moyenne_mb (même problème probable)
facteur_moy = df_sdv['data_moyenne_mb'].mean() / synthetic_df['data_moyenne_mb'].mean()
synthetic_df['data_moyenne_mb'] = (
    synthetic_df['data_moyenne_mb'] * facteur_moy
).clip(lower=0)

# Vérification finale
print("\n=== COMPARAISON FINALE ===")
cols_a_verifier = [
    'churn', 'data_totale_mb', 'data_moyenne_mb',
    'recence_cdr_jours', 'nb_appels', 'satisfaction_client'
]
for col in cols_a_verifier:
    if col in synthetic_df.columns and col in df_sdv.columns:
        orig = df_sdv[col].mean()
        synt = synthetic_df[col].mean()
        ecart = abs(orig - synt) / (orig + 1e-9) * 100
        statut = "✓" if ecart < 15 else "⚠️"
        print(f"{statut} {col:<35} orig={orig:.2f}  synt={synt:.2f}  écart={ecart:.1f}%")

Facteur de correction : 1.000

=== COMPARAISON FINALE ===
✓ churn                               orig=0.27  synt=0.27  écart=2.4%
✓ data_totale_mb                      orig=49.39  synt=49.39  écart=0.0%
✓ data_moyenne_mb                     orig=4.70  synt=4.70  écart=0.0%
✓ recence_cdr_jours                   orig=109.07  synt=108.21  écart=0.8%
⚠️ nb_appels                           orig=5.89  synt=3.45  écart=41.3%
✓ satisfaction_client                 orig=2.47  synt=2.60  écart=5.2%


In [99]:
import numpy as np

facteur = df_sdv['data_totale_mb'].mean() / synthetic_df['data_totale_mb'].mean()
synthetic_df['data_totale_mb'] = (synthetic_df['data_totale_mb'] * facteur).clip(lower=0)

facteur_moy = df_sdv['data_moyenne_mb'].mean() / synthetic_df['data_moyenne_mb'].mean()
synthetic_df['data_moyenne_mb'] = (synthetic_df['data_moyenne_mb'] * facteur_moy).clip(lower=0)

print("=== VALIDATION FINALE SDV ===")
for col in ['churn', 'data_totale_mb', 'recence_cdr_jours', 'nb_appels']:
    orig = df_sdv[col].mean()
    synt = synthetic_df[col].mean()
    ecart = abs(orig - synt) / (orig + 1e-9) * 100
    statut = "✓" if ecart < 15 else "⚠️"
    print(f"{statut} {col:<30} orig={orig:.1f}  synt={synt:.1f}  écart={ecart:.0f}%")

=== VALIDATION FINALE SDV ===
✓ churn                          orig=0.3  synt=0.3  écart=2%
✓ data_totale_mb                 orig=49.4  synt=49.4  écart=0%
✓ recence_cdr_jours              orig=109.1  synt=108.2  écart=1%
⚠️ nb_appels                      orig=5.9  synt=3.5  écart=41%


In [100]:
# ── Corrections post-génération ──────────────────────────────────────────────

# 1. Recalibrer nb_appels
facteur_appels = df_sdv['nb_appels'].mean() / synthetic_df['nb_appels'].mean()
synthetic_df['nb_appels'] = (
    synthetic_df['nb_appels'] * facteur_appels
).clip(lower=0).round()

# 2. Recréer client_id propre (1 → 300)
synthetic_df['client_id'] = range(1, len(synthetic_df) + 1)

# 3. Imputer les NaN GPS par la médiane du dataset original
synthetic_df['latitude_moyenne'] = synthetic_df['latitude_moyenne'].fillna(
    df_sdv['latitude_moyenne'].median()
)
synthetic_df['longitude_moyenne'] = synthetic_df['longitude_moyenne'].fillna(
    df_sdv['longitude_moyenne'].median()
)

# ── Validation finale complète ────────────────────────────────────────────────
print("=== VALIDATION FINALE COMPLÈTE ===")
cols = ['churn', 'data_totale_mb', 'recence_cdr_jours',
        'nb_appels', 'satisfaction_client', 'nb_reclamations']
for col in cols:
    if col in synthetic_df.columns:
        orig = df_sdv[col].mean()
        synt = synthetic_df[col].mean()
        ecart = abs(orig - synt) / (orig + 1e-9) * 100
        statut = "✓" if ecart < 20 else "⚠️"
        print(f"{statut} {col:<30} orig={orig:.1f}  synt={synt:.1f}  écart={ecart:.0f}%")

print(f"\nNaN restants :")
nan_cols = synthetic_df.isnull().sum()
print(nan_cols[nan_cols > 0])
print(f"\nclient_id range : {synthetic_df['client_id'].min()} → {synthetic_df['client_id'].max()}")

=== VALIDATION FINALE COMPLÈTE ===
✓ churn                          orig=0.3  synt=0.3  écart=2%
✓ data_totale_mb                 orig=49.4  synt=49.4  écart=0%
✓ recence_cdr_jours              orig=109.1  synt=108.2  écart=1%
✓ nb_appels                      orig=5.9  synt=5.9  écart=0%
✓ satisfaction_client            orig=2.5  synt=2.6  écart=5%
✓ nb_reclamations                orig=3.1  synt=2.8  écart=11%

NaN restants :
facture_moyenne_mensuelle     24
satisfaction_client           30
date_consentement            122
data_totale_mb                18
nb_reclamations               50
tendance_data_pct              9
ratio_sms_appels               9
ratio_data_voix               33
score_frustration             54
dtype: int64

client_id range : 1 → 300


In [101]:
# ── Export final du dataset SDV ───────────────────────────────────────────────
synthetic_df.to_csv(
    r"C:/Users/Paul Adrien/Desktop/dataset_sdv_final.csv",
    index=False,
    encoding="utf-8-sig"
)
print(f"✓ Export réussi")
print(f"  Lignes    : {len(synthetic_df)}")
print(f"  Colonnes  : {len(synthetic_df.columns)}")
print(f"  Churn     : {synthetic_df['churn'].mean():.1%}")

✓ Export réussi
  Lignes    : 300
  Colonnes  : 38
  Churn     : 26.7%


In [105]:
df_biaise = synthetic_df.copy()

# ── Biais 1 : data faible → churn probable ───────────────────────────────────
seuil_data = df_biaise['data_totale_mb'].quantile(0.15)
mask_data = (
    (df_biaise['data_totale_mb'].fillna(0) < seuil_data) &
    (df_biaise['data_totale_mb'].fillna(0) > 0) &
    (df_biaise['churn'] == 0)
)
idx_b1 = df_biaise[mask_data].sample(frac=0.30, random_state=100).index
df_biaise.loc[idx_b1, 'churn'] = 1
print(f"Biais 1 : {len(idx_b1)} clients à faible data → churn=1 forcé")

# ── Biais 2 : frustration élevée → churn probable ────────────────────────────
seuil_frust = df_biaise['score_frustration'].quantile(0.80)
mask_frust = (
    (df_biaise['score_frustration'].fillna(0) >= seuil_frust) &
    (df_biaise['churn'] == 0)
)
idx_b2 = df_biaise[mask_frust].sample(frac=0.25, random_state=101).index
df_biaise.loc[idx_b2, 'churn'] = 1
print(f"Biais 2 : {len(idx_b2)} clients frustrés → churn=1 forcé")

# ── Biais 3 : RURAL + signal Faible → churn probable ─────────────────────────
mask_reseau = (
    (df_biaise['zone_reseau_principale'] == 'RURAL') &
    (df_biaise['qualite_signal_dominante'] == 'Faible') &
    (df_biaise['churn'] == 0)
)
n_rural = mask_reseau.sum()
if n_rural > 0:
    idx_b3 = df_biaise[mask_reseau].sample(frac=0.40, random_state=102).index
    df_biaise.loc[idx_b3, 'churn'] = 1
    print(f"Biais 3 : {len(idx_b3)} clients rural+signal faible → churn=1 forcé")
else:
    print(f"Biais 3 : aucun client RURAL+Faible dans ce tirage — ignoré")

# ── Résultat ──────────────────────────────────────────────────────────────────
print(f"\nTaux de churn avant biais : {synthetic_df['churn'].mean():.1%}")
print(f"Taux de churn après biais : {df_biaise['churn'].mean():.1%}")
print(f"Clients basculés en churn : {(df_biaise['churn'] - synthetic_df['churn']).clip(lower=0).sum():.0f}")

Biais 1 : 7 clients à faible data → churn=1 forcé
Biais 2 : 7 clients frustrés → churn=1 forcé
Biais 3 : 0 clients rural+signal faible → churn=1 forcé

Taux de churn avant biais : 26.7%
Taux de churn après biais : 31.3%
Clients basculés en churn : 14


In [106]:
# ── Export final ML-ready ─────────────────────────────────────────────────────
df_biaise.to_csv(
    r"C:/Users/Paul Adrien/Desktop/dataset_sdv_final.csv",
    index=False,
    encoding="utf-8-sig"
)
print(f"✓ Export réussi")
print(f"  Lignes   : {len(df_biaise)}")
print(f"  Colonnes : {len(df_biaise.columns)}")
print(f"  Churn    : {df_biaise['churn'].mean():.1%}")

✓ Export réussi
  Lignes   : 300
  Colonnes : 38
  Churn    : 31.3%
